# NOVA 01 — MOD-01 Obstacle Detection (YOLOv8n)
**Attach datasets** (Add Data, right sidebar):
- `jhontroya/dectectra-dataset`
- `kushagrapandya/visdrone-dataset`

**Accelerator:** GPU T4 x2 or P100. Full training ~6-9h — fits one Kaggle session (12h limit). Enable *Persistence: Files* to survive restarts.

In [ ]:
# ── NOVA bootstrap: clone repo + resolve HF token from Kaggle secret ──
# Before running: Add-ons > Secrets > attach a secret named HF_TOKEN
# (your HuggingFace write token). Settings > Accelerator > GPU T4/P100.
import os, sys, subprocess

REPO = 'https://github.com/BertinAm/nova-ml.git'
if not os.path.exists('/kaggle/working/nova-ml'):
    subprocess.run(['git', 'clone', REPO, '/kaggle/working/nova-ml'], check=True)
os.chdir('/kaggle/working/nova-ml')
sys.path.insert(0, '/kaggle/working/nova-ml/scripts')

from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['NOVA_HF_USERNAME'] = 'unixio'
print('Bootstrap OK — repo cloned, HF token loaded.')

In [ ]:
!pip install -q ultralytics onnx2tf onnx

In [ ]:
# Inspect what the attached datasets actually look like (layouts vary)
!ls /kaggle/input/
!find /kaggle/input/dectectra-dataset -maxdepth 2 -type d | head -20

In [ ]:
!python scripts/prepare_obstacle_dataset.py \
    --detectra /kaggle/input/dectectra-dataset \
    --visdrone /kaggle/input/visdrone-dataset \
    --out /kaggle/working/datasets/obstacle_combined

In [ ]:
# Point the data config at the merged dataset.
# NOTE: verify/fix the class list in configs/obstacle_data.yaml against
# the Detectra data.yaml before a full run (see the config's comments).
import yaml
cfg = yaml.safe_load(open('configs/obstacle_data.yaml'))
cfg['path'] = '/kaggle/working/datasets/obstacle_combined'
yaml.dump(cfg, open('configs/obstacle_data.yaml', 'w'))
print(cfg['path'], '| nc =', cfg['nc'])

In [ ]:
# Full training + TFLite INT8 export (Ultralytics native export)
!python scripts/train_obstacle.py --data configs/obstacle_data.yaml \
    --epochs 100 --imgsz 320 --batch 32

In [ ]:
# Publish to HuggingFace: unixio/nova-obstacle-detection
import glob
tflite = glob.glob('/kaggle/working/runs/obstacle_student/weights/*_int8.tflite') + \
         glob.glob('/kaggle/working/runs/obstacle_student/weights/best_saved_model/*_int8.tflite')
print('TFLite candidates:', tflite)
best_pt = '/kaggle/working/runs/obstacle_student/weights/best.pt'
!python scripts/push_to_huggingface.py --module MOD-01 \
    --pytorch {best_pt} --tflite {tflite[0]} \
    --eval-json /kaggle/working/evaluation/obstacle_results.json --version 1.0.0